# 05 — The same task looks emergent under exact-match and smooth under edit-distance

**Programme 05 — Emergence and Its Discontents (and Reasoning).** [Programme file.](../programmes/05-emergence-and-reasoning.md)

By the end of this notebook you will have:
- evaluated a small open-model family at multiple scales on a controlled task (3- and 4-digit reverse-arithmetic),
- recorded *three* metrics — exact-match accuracy, mean token-edit-distance, and log-probability of the correct continuation,
- plotted each against scale,
- and seen the qualitative Schaeffer, Miranda, Koyejo (2023, [arXiv:2304.15004](https://arxiv.org/abs/2304.15004)) finding: under a continuous metric, the apparent discontinuity of the exact-match curve disappears or shrinks.

What this notebook is *not*: a refutation of the strong-emergence claim. Schaeffer et al. ran this argument at much larger scale with multiple model families and tasks. The notebook is a budget illustration; the reading-direction is the lesson, not the magnitudes.

In [ ]:
# Install pinned versions (Colab).
# %pip install -q torch==2.4.1 transformers==4.45.2 numpy==1.26.4 matplotlib==3.9.2

In [ ]:
import math, os, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 11
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
os.makedirs('figures/05', exist_ok=True)

MODELS = [
    ('EleutherAI/pythia-70m',  70e6),
    ('EleutherAI/pythia-160m', 160e6),
    ('EleutherAI/pythia-410m', 410e6),
    ('EleutherAI/pythia-1.4b', 1.4e9),
]

## A controlled task with discrete and continuous metrics

*Reverse-string:* given a digit string, output it reversed. Exact-match is the *natural* discrete metric (binary correct / incorrect). Token-edit-distance from the gold reversal is a *continuous* metric on the same outputs: a model that gets 4/5 digits right gets a non-trivial score.

**Prediction (Schaeffer et al.).** Exact-match should be near-zero for the smallest models and improve sharply at scale. Edit-distance — *and* log-probability of the correct continuation — should improve smoothly across all scales.

In [ ]:
rng = np.random.default_rng(SEED)
DIGITS_LEN = 4
N_EVAL = 32

def make_examples(n=N_EVAL, length=DIGITS_LEN):
    items = []
    for _ in range(n):
        s = ''.join(str(d) for d in rng.integers(0, 10, size=length))
        items.append((s, s[::-1]))
    return items

FEWSHOT = (
    'Reverse the digit string.\n'
    'Input: 31415\nOutput: 51413\n'
    'Input: 27182\nOutput: 28172\n'
    'Input: 16180\nOutput: 08161\n'
)

def make_prompt(x: str) -> str:
    return FEWSHOT + f'Input: {x}\nOutput:'

EVAL = make_examples()
print(EVAL[:3])

In [ ]:
def token_edit_distance(a: str, b: str) -> int:
    # Levenshtein distance at the *character* level (digits are tokens for our task).
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev = dp[0]; dp[0] = i
        for j, cb in enumerate(b, 1):
            tmp = dp[j]
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + (ca != cb))
            prev = tmp
    return dp[-1]

In [ ]:
@torch.no_grad()
def eval_model(model_name, eval_set, max_new=DIGITS_LEN + 2):
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.pad_token = tok.eos_token if tok.pad_token is None else tok.pad_token
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32).to(DEVICE).eval()
    em = 0; ed_sum = 0; logp_sum = 0.0
    for x, y in eval_set:
        prompt = make_prompt(x)
        ids = tok(prompt, return_tensors='pt').input_ids.to(DEVICE)
        # generate
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        gen = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip().split('\n')[0]
        # parse continuation digits
        pred = ''.join(ch for ch in gen if ch.isdigit())[:DIGITS_LEN]
        em += int(pred == y)
        ed_sum += token_edit_distance(pred, y) / DIGITS_LEN
        # log-probability of the gold continuation (teacher-forced)
        full = prompt + ' ' + y
        ids_full = tok(full, return_tensors='pt').input_ids.to(DEVICE)
        n_prompt = tok(prompt, return_tensors='pt').input_ids.shape[1]
        logits = model(ids_full).logits  # (1, T, V)
        log_probs = logits.log_softmax(dim=-1)
        # log p(target tokens | prompt)
        tgt_ids = ids_full[0, n_prompt:]
        lp = log_probs[0, n_prompt - 1:-1, :].gather(-1, tgt_ids.unsqueeze(-1)).squeeze(-1)
        logp_sum += lp.sum().item() / max(1, len(tgt_ids))
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return em / len(eval_set), ed_sum / len(eval_set), logp_sum / len(eval_set)

rows = []
for name, n_params in MODELS:
    em, ed, lp = eval_model(name, EVAL)
    rows.append((name, n_params, em, ed, lp))
    print(f'{name:<32s}  EM={em:.2f}  edit/L={ed:.2f}  logp={lp:+.3f}')

In [ ]:
names = [r[0].split('/')[-1] for r in rows]
scales = np.array([r[1] for r in rows])
em = np.array([r[2] for r in rows])
ed = np.array([r[3] for r in rows])
lp = np.array([r[4] for r in rows])

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, y, ylabel in zip(axes, [em, 1 - ed, lp], ['exact-match acc', '1 - mean edit / L', 'mean log-prob of gold']):
    ax.plot(scales, y, 'o-')
    for s, v, nm in zip(scales, y, names):
        ax.annotate(nm, (s, v), fontsize=7, xytext=(4, 4), textcoords='offset points')
    ax.set_xscale('log'); ax.set_xlabel('parameters (log)'); ax.set_ylabel(ylabel)
fig.suptitle('Reverse-string task: discrete vs. continuous metrics across scale')
fig.tight_layout()
fig.savefig('figures/05/metrics_vs_scale.png', dpi=120)
plt.show()

## What this shows. What this does not show. What would refute it.

**What this shows.** On this small task at small scales, the exact-match curve is more discontinuous than the edit-distance curve and far more discontinuous than the gold-log-prob curve. The qualitative shape is what Schaeffer et al. (2023) argue is general: nonlinear / threshold metrics manufacture apparent emergence; continuous metrics reveal smooth scaling underneath. The fact that exact-match looks emergent on this tiny scale is not evidence of *real* emergence — it is the artifact Schaeffer et al. predict.

**What this does not show.** A handful of Pythia checkpoints on one tiny task is not the Schaeffer et al. evidence base. The serious argument requires multiple model families and a broad task suite. We have also not engaged with the harder cases: tasks where it is *not obvious* that a continuous relaxation exists, and tasks where the discrete capability is the thing you actually care about.

**What would refute the broader Schaeffer et al. claim.** A task where the most natural continuous metric *also* shows a sharp transition at scale — and where the transition is not predictable from extrapolation of smaller-scale runs. Such tasks may exist; the Schaeffer et al. argument is about the canonical *Wei et al. catalog*, not about all possible tasks.

**Where to go next.** Read Schaeffer, Miranda, Koyejo (2023) in full. The headline argument is on a slide; the substance is in the per-metric, per-task figures.